# Step 10: Engagement Intelligence Analysis

## Overview
This notebook performs non-ML workforce engagement analysis:
1. Aggregate engagement, satisfaction, and work-life balance scores by Department.
2. Identify lowest-engagement employees across the organization.
3. Export summary insights to `data/processed/engagement_intelligence_summary.csv`.


In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join("..", "data", "processed")

perf_df = pd.read_csv(os.path.join(PROCESSED_DIR, "engagement_processed.csv"))
attr_df = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_attrition_processed.csv"))

print(f"Engagement dataset shape: {perf_df.shape}")
print(f"Attrition dataset shape: {attr_df.shape}")


Engagement dataset shape: (2845, 28)
Attrition dataset shape: (1470, 32)


---
## 1. Department-Level Engagement Aggregation


In [2]:
# Standardize Department column
dept_col = 'DepartmentType' if 'DepartmentType' in perf_df.columns else 'Department'

dept_summary = perf_df.groupby(dept_col).agg(
    Employee_Count=('Employee ID', 'count'),
    Avg_Engagement=('Engagement Score', 'mean'),
    Avg_Satisfaction=('Satisfaction Score', 'mean'),
    Avg_WorkLifeBalance=('Work-Life Balance Score', 'mean'),
    Avg_Performance_Rating=('Current Employee Rating', 'mean')
).reset_index().sort_values(by='Avg_Engagement', ascending=True)

print("=== Engagement & Satisfaction Summary by Department ===")
print(dept_summary.to_string(index=False))


=== Engagement & Satisfaction Summary by Department ===
      DepartmentType  Employee_Count  Avg_Engagement  Avg_Satisfaction  Avg_WorkLifeBalance  Avg_Performance_Rating
          Production            1910        2.910471          3.031414             2.971728                2.989005
       Admin Offices              79        2.949367          2.506329             3.202532                3.025316
Software Engineering             112        2.955357          3.098214             2.991071                2.901786
               Sales             311        2.983923          3.135048             3.028939                2.922830
               IT/IS             409        3.024450          3.012225             2.980440                2.968215
    Executive Office              24        3.375000          3.083333             3.291667                2.791667


---
## 2. Lowest-Engagement Employees Identification


In [3]:
# Composite engagement index (scale 1-5)
perf_df['Composite_Engagement_Index'] = (perf_df['Engagement Score'] + perf_df['Satisfaction Score'] + perf_df['Work-Life Balance Score']) / 3.0

low_engagement_threshold = perf_df['Composite_Engagement_Index'].quantile(0.10)
lowest_engaged_emp = perf_df[perf_df['Composite_Engagement_Index'] <= low_engagement_threshold].sort_values(by='Composite_Engagement_Index')

print(f"Low engagement threshold (Bottom 10%): {low_engagement_threshold:.2f}")
print(f"Identified {len(lowest_engaged_emp)} low-engagement employees.")
print("\nSample Low-Engagement Employees:")
print(lowest_engaged_emp[['Employee ID', dept_col, 'Title', 'Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score', 'Composite_Engagement_Index']].head(10).to_string(index=False))

out_path = os.path.join(PROCESSED_DIR, "engagement_intelligence_summary.csv")
dept_summary.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")


Low engagement threshold (Bottom 10%): 2.00
Identified 469 low-engagement employees.

Sample Low-Engagement Employees:
 Employee ID DepartmentType                    Title  Engagement Score  Satisfaction Score  Work-Life Balance Score  Composite_Engagement_Index
        3616     Production  Production Technician I                 1                   1                        1                         1.0
        3140     Production  Production Technician I                 1                   1                        1                         1.0
        3798     Production  Production Technician I                 1                   1                        1                         1.0
        3811          IT/IS     Sr. Network Engineer                 1                   1                        1                         1.0
        3938          IT/IS             Data Analyst                 1                   1                        1                         1.0
        3982     